# Análise de Fourier com Python

## 1. Transformada Discreta de Fourier (DFT)

A DFT é uma técnica fundamental que converte um sinal do domínio do tempo para o domínio da frequência. Ela assume que o sinal é periódico, repetindo-se infinitamente.

### Definição Matemática

**DFT (Análise):**
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j\frac{2\pi}{N}kn}, \quad k = 0, 1, ..., N-1$$

Onde:
*   $n$: índice temporal discreto
*   $k$: índice de frequência discreto
*   $N$: número de amostras
*   $f_k = k \cdot \frac{F_s}{N}$: frequência correspondente ao índice $k$ (Hz)
*   $F_s$: frequência de amostragem

**DFT Inversa (Síntese):**
$$x[n] = \frac{1}{N}\sum_{k=0}^{N-1} X[k] e^{j\frac{2\pi}{N} kn}, \quad n = 0, 1, ..., N-1$$

### Representação Matricial

Definindo $W_N = e^{-j2\pi/N}$, a DFT pode ser expressa como uma operação linear:

$$\mathbf{X} = \mathbf{W} \mathbf{x}$$

Onde $\mathbf{W}$ é a matriz DFT de dimensão $N \times N$:

$$\mathbf{W} = \begin{bmatrix}
1 & 1 & 1 & \cdots & 1 \\
1 & W_N^{1} & W_N^{2} & \cdots & W_N^{N-1} \\
1 & W_N^{2} & W_N^{4} & \cdots & W_N^{2(N-1)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & W_N^{N-1} & W_N^{2(N-1)} & \cdots & W_N^{(N-1)(N-1)}
\end{bmatrix}$$


## 2. Implementações da DFT

Vamos implementar e comparar diferentes formas de calcular a DFT, destacando a importância do algoritmo FFT (Fast Fourier Transform).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import timeit
import scipy.signal as signal

In [ ]:
# Sinal de teste com valores aleatórios
x = np.random.random(32)

In [ ]:
x

### 2.1 Implementação Força Bruta

Esta implementação segue diretamente a definição matemática da DFT, com complexidade computacional $\mathcal{O}(N^2)$

In [ ]:
N = x.shape[0]
N

In [ ]:
np.arange(N)

In [ ]:
X = np.zeros(N, dtype=complex)

In [ ]:
def DFT_loop(x):
    N = len(x)
    K = N
    X = np.zeros(N, dtype=complex)
    for k in range(K):          # Itera sobre as raias
        for n in range(N):      # Itera sobre as amostras 
            X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
    return X

### 2.2 Implementação Vetorizada

Utiliza vetorização do NumPy para uma implementação mais eficiente.

In [ ]:
def DFT_matrix(x):
    N = len(x)
    n = np.arange(N)
    k = n.reshape(N, 1)  # Vetor coluna para broadcasting
    # Matriz DFT: W_{nk} = exp(-2jπ kn/N)
    W = np.exp(-2j * np.pi * k * n / N)
    return np.dot(x, W)

### 2.3 Transformada Rápida de Fourier (FFT) e Comparação

O algoritmo FFT, implementado em bibliotecas como `numpy.fft`, reduz a complexidade para $\mathcal{O}(N \log_2 N)$, explorando simetrias e redundâncias nos cálculos.

In [ ]:
# Verificação de equivalência entre as implementações
X1 = DFT_loop(x)
X2 = DFT_matrix(x)
X_fft = np.fft.fft(x)

In [ ]:
np.allclose(X1, X_fft)

In [ ]:
np.allclose(X2, X_fft)

In [ ]:
# Comparação de desempenho
print("\n--- Comparação de Desempenho ---")
print(f"DFT_loop: {timeit.timeit('DFT_loop(x)', globals=globals(), number=1000)/1000:.8f} ms")
print(f"DFT_matrix: {timeit.timeit('DFT_matrix(x)', globals=globals(), number=1000)/1000:.8f} ms")
print(f"np.fft.fft: {timeit.timeit('np.fft.fft(x)', globals=globals(), number=1000)/1000:.8f} ms")

**Observação sobre desempenho:**

Este ganho torna a FFT essencial para processamento de sinais em tempo real e análise de grandes volumes de dados.

**Aplicação em convolução:** A FFT permite implementar convoluções $\mathcal{O}[N^2]$  de forma eficiente:

\begin{equation*}
X[k] = \text{DFT}\{x[n]\}, \quad Y[k] = \text{DFT}\{y[n]\}
\end{equation*}

\begin{equation*}
Z[k] = X[k] \cdot Y[k]
\end{equation*}

\begin{equation*}
(x * y)[n] = \text{IDFT}\{Z[k]\}
\end{equation*}

* response = signal.fftconvolve(x, y, mode='full')

## 3. Análise de Sinais Periódicos

Vamos analisar um sinal senoidal puro para entender as propriedades do espectro DFT.

In [ ]:
f = 100          # Frequência do sinal (Hz)
fs = 1000        # Taxa de amostragem (Hz)
Td = 0.01         # Duração do sinal (s) 

t = np.arange(0, Td, 1/fs)
x = np.sin(2 * np.pi * f * t)

# Visualização no domínio do tempo
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, x, 'b-', linewidth=1.5)
ax.set_xlabel('Tempo [s]')
ax.set_ylabel('Amplitude')
ax.set_title(f'Sinal Senoidal de {f} Hz amostrado a {fs} Hz')
ax.grid(alpha=0.3)
plt.show()

In [ ]:
x.shape

### Propriedade de Simetria do Espectro

Para sinais reais, os coeficientes da DFT apresentam simetria conjugada:

\begin{equation*}
X[k] = X^*[(N - k) \bmod N]
\end{equation*}

Isso implica que a magnitude do espectro é simétrica:

\begin{equation*}
|X[k]| = |X[N - k]|
\end{equation*}

e a fase é anti-simétrica:

\begin{equation*}
\angle X[k] = -\angle X[N - k]
\end{equation*}

In [ ]:
X = np.fft.fft(x)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8))

mag = np.abs(X)
ax[0].stem(mag)
ax[0].set_xlabel('$k$')
ax[0].set_ylabel('Magnitude')
ax[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

phase = np.angle(X)
ax[1].stem(phase)
ax[1].set_xlabel('$k$')
ax[1].set_ylabel('Phase [rad]')
ax[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

### Exemplo ($N = 10$)

$$
X[k] = X^*[(N - k) \bmod N]
$$

$k = 0$: $X[0]$ (real)  
$k = 1$: $X[1] = X^*[9]$  
$k = 2$: $X[2] = X^*[8]$  
$k = 3$: $X[3] = X^*[7]$  
$k = 4$: $X[4] = X^*[6]$  
$k = 5$: $X[5] = X^*[5]$ (real, Nyquist)  
$k = 6$: $X[6] = X^*[4]$  
$k = 7$: $X[7] = X^*[3]$  
$k = 8$: $X[8] = X^*[2]$  
$k = 9$: $X[9] = X^*[1]$  

**Valores reais:**
$k = 0$ e $k = N/2 = 5$

E para $N = 9$?

<div align="center">
  <img src="img/fft.png" alt="title" width="480">
</div>

In [ ]:
# Espectro completo
freqs = np.fft.fftfreq(len(x), 1/fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 8))

# Magnitude
ax[0].stem(freqs, np.abs(X))
ax[0].set_xlabel('Frequência [Hz]')
ax[0].set_ylabel('Magnitude')
ax[0].set_title('Espectro de Magnitude')

# Fase
ax[1].stem(freqs, np.angle(X))
ax[1].set_xlabel('Frequência [Hz]')
ax[1].set_ylabel('Fase [rad]')
ax[1].set_title('Espectro de Fase')

plt.tight_layout()
plt.show()

Devido à propriedade de simetria da magnitude dos coeficientes da DFT, é necessário considerar apenas os coeficientes com índices $k = 0, \ldots, \left[\frac{N}{2} + 1\right]$, para $N$ par, ou $k = 0, \ldots, \left[\frac{N-1}{2}\right]$, para $N$ ímpar.

In [ ]:
# Analisando apenas a metade positiva do espectro
n_pos = len(X) // 2 + 1
freqs_pos = np.arange(0, n_pos) * (fs / len(X))
mag_pos = np.abs(X[:n_pos])

fig, ax = plt.subplots(figsize=(10, 4))
ax.stem(freqs_pos, mag_pos)
ax.set_xlabel('Frequência [Hz]')
ax.set_ylabel('Magnitude')
ax.set_title('Espectro de Magnitude (DC, frequências positivas e de Nyquist)')

plt.tight_layout()
plt.show()

## 4. Resolução Espectral

A resolução em frequência da DFT é determinada pela duração do sinal analisado:

$$\Delta f = \frac{f_s}{N} = \frac{1}{T_d}$$

no qual $T_d$ é a duração da janela em segundos. Quanto maior a duração do sinal, melhor a resolução espectral. 

Testar $T_d = 0.02$.

## 5. Zero Padding

*Zero padding* é a técnica de adicionar zeros ao final do sinal antes de calcular a FFT. Isso aumenta o número de pontos no espectro.

In [ ]:
f = 100          # Frequência do sinal (Hz)
fs = 1000        # Taxa de amostragem (Hz)
N_original = 40  # 40 amostras (0.04 segundos)

# Sinal com zero padding
t = np.arange(0, N_original/fs, 1/fs)
x = np.sin(2 * np.pi * f * t)
x_pad = np.hstack([x, np.zeros(10)])  # Adiciona 10 zeros ao final
t_pad = np.arange(0, (N_original+10)/fs, 1/fs)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8))

# Sinal original - linha + pontos
ax[0].plot(t, x, 'b-', linewidth=1.5, alpha=0.7)
ax[0].set_xlabel('Tempo [s]')
ax[0].set_ylabel('Amplitude')
ax[0].set_title(f'Sinal Original - {N_original} amostras')

# Sinal com zero padding - linha + pontos
ax[1].plot(t_pad, x_pad, 'r-', linewidth=1.5, alpha=0.7)
ax[1].set_xlabel('Tempo [s]')
ax[1].set_ylabel('Amplitude')
ax[1].set_title(f'Sinal com Zero Padding - {len(x_pad)} amostras')
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Comparação dos espectros
X_original = np.fft.fft(x)
X_pad = np.fft.fft(x_pad)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Sem zero padding
freqs1 = np.fft.fftfreq(len(x), 1/fs)
ax1.stem(freqs1[:len(x)//2], np.abs(X_original[:len(x)//2]))
ax1.set_xlabel('Frequência [Hz]')
ax1.set_ylabel('Magnitude')
ax1.set_title(f'Espectro sem zero padding ({len(x)} pontos)')


# Com zero padding
freqs2 = np.fft.fftfreq(len(x_pad), 1/fs)
ax2.stem(freqs2[:len(x_pad)//2], np.abs(X_pad[:len(x_pad)//2]))
ax2.set_xlabel('Frequência [Hz]')
ax2.set_ylabel('Magnitude')
ax2.set_title(f'Espectro com zero padding ({len(x_pad)} pontos)')

plt.tight_layout()
plt.show()

**Observação:** O *zero padding* aumenta o número de pontos no espectro, proporcionando uma visualização mais suave, mas não melhora a separação entre frequências próximas (resolução espectral).

## 6. Vazamento Espectral (Spectral Leakage)

Quando a duração do sinal não contém um número inteiro de períodos da componente senoidal, ocorre o **vazamento espectral**: a energia do sinal se espalha para frequências adjacentes, distorcendo o espectro.

In [ ]:
f = 100          # Frequência do sinal (Hz)
fs = 1000        # Taxa de amostragem (Hz)
Td = 0.687

t = np.arange(0, Td, 1/fs)
x = np.sin(2 * np.pi * f * t)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, x)
ax.set_xlabel('Tempo [s]')
ax.set_ylabel('Amplitude')
ax.set_title('Sinal com número não inteiro de períodos')

plt.tight_layout()
plt.show()

In [ ]:
X = np.fft.fft(x)

mag = np.abs(X[0:int((X.shape[0]-1)/2)])
freqs = np.arange(0, int((X.shape[0]-1)/2))*1/Td

fig, ax = plt.subplots(figsize=(10, 4))
ax.stem(freqs,mag)
ax.set_xlabel('Frequência [Hz]')
ax.set_ylabel('Magnitude')
ax.set_title('Espectro com vazamento espectral')

plt.tight_layout()
plt.show()

### Solução: Janelamento

O janelamento consiste em multiplicar o sinal por uma janela (função de ponderação) que atenua as descontinuidades nas bordas, reduzindo o vazamento espectral. A escolha da janela envolve um compromisso entre:

1. **Lóbulo principal estreito**: melhor resolução
2. **Lóbulos laterais baixos**: menos vazamento espectral

Janelas comuns incluem Hamming, Hann, Blackman, entre outras.

In [ ]:
from scipy import signal

# Aplicando janela de Hamming
w = signal.get_window('hamming', len(x))
x_windowed = x * w

# Plot comparativo
fig, ax = plt.subplots(3, 1, figsize=(10, 10))

# Sinal original
ax[0].plot(t, x)
ax[0].set_xlabel('Tempo [s]')
ax[0].set_ylabel('Amplitude')
ax[0].set_title('Sinal Original')

# Janela de Hamming
ax[1].plot(t, w, 'g-')
ax[1].set_xlabel('Tempo [s]')
ax[1].set_ylabel('Amplitude')
ax[1].set_title('Janela de Hamming')
ax[1].grid(True, alpha=0.3)

# Sinal janelado
ax[2].plot(t, x_windowed, 'r-')
ax[2].set_xlabel('Tempo [s]')
ax[2].set_ylabel('Amplitude')
ax[2].set_title('Sinal Janelado (Original × Janela)')
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
X = np.fft.fft(x)

mag = np.abs(X[0:int((X.shape[0]-1)/2)])
freqs = np.arange(0, int((X.shape[0]-1)/2))*1/Td

fig, ax = plt.subplots(figsize=(10, 4))
ax.stem(freqs,mag)
ax.set_xlabel('Frequência [Hz]')
ax.set_ylabel('Magnitude')
ax.set_title('Espectro com vazamento espectral')

plt.tight_layout()
plt.show()

In [ ]:
Xwin = np.fft.fft(x_windowed)

fig, ax = plt.subplots(figsize=(10, 4))
mag = np.abs(Xwin[0:int((Xwin.shape[0]-1)/2)])
freqs = np.arange(0, int((Xwin.shape[0]-1)/2))*1/Td
ax.stem(freqs,mag)
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude')

plt.tight_layout()
plt.show()

**Observação:** O janelamento reduz significativamente os lóbulos laterais (vazamento espectral), mas alarga ligeiramente o lóbulo principal (reduz a resolução).

## 7. Frequência de amostragem

In [ ]:
f = 600  # Frequency, in cycles per second, or Hertz
fs = 1000  # Sampling rate, or number of measurements per second

t = np.arange(0, 0.04, 1/fs)
x = np.sin(2 * np.pi * f * t)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, x)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Amplitude');

plt.tight_layout()
plt.show()

In [ ]:
X = np.fft.fft(x)

freqs = np.fft.fftfreq(len(x), 1/fs)

fig, ax = plt.subplots(figsize=(10, 4))

ax.stem(freqs, np.abs(X))
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude')

plt.tight_layout()
plt.show()

## 8. Reconstrução da FFT

A DFT e sua inversa (IDFT) formam um par de transformadas que permitem reconstrução perfeita do sinal original, sem perda de informação.

In [ ]:
x = np.random.random(16)  # Sinal aleatório de teste
X = np.fft.fft(x)        # Transformada direta
x_hat = np.fft.ifft(X)  # Transformada inversa

In [ ]:
np.allclose(x, x_hat)

In [ ]:
np.sum(x - x_hat)

## Referências

LIBROSA.  *Librosa: Audio and Music Signal Analysis*.  Disponível em: https://librosa.org/. 

QUATIERI, Thomas F.  
*Discrete-Time Speech Signal Processing: Principles and Practice*.  Upper Saddle River: Prentice-Hall PTR, 2002.

RUMSEY, Francis.  *Spatial Audio*.  Oxford: Focal Press, 2012.

SCIPY.  *Elegant SciPy: Capítulo 4*.  Disponível em: https://www.oreilly.com/library/view/elegant-scipy/9781491922927/ch04.html. Acesso em: 24 mar. 2026.
